## 데이터 로드, 형식 정리

In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path('data/raw/SeoulBikeData.csv')
CLEAN_PATH = Path('data/processed/seoul_bike_clean.csv')

# 원본은 UTF-8이 아님
df = pd.read_csv(RAW_PATH, encoding='latin-1')
print(df.shape)
df.head()

(8760, 14)


,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [2]:
# 컬럼명을 영문 소문자로 변경 (°, 공백, 괄호, 단위 제거)
COLUMN_NAMES = {
    'Date': 'date',
    'Rented Bike Count': 'rented_bike_count',
    'Hour': 'hour',
    'Temperature(°C)': 'temperature',
    'Humidity(%)': 'humidity',
    'Wind speed (m/s)': 'wind_speed',
    'Visibility (10m)': 'visibility',
    'Dew point temperature(°C)': 'dew_point_temp',
    'Solar Radiation (MJ/m2)': 'solar_radiation',
    'Rainfall(mm)': 'rainfall',
    'Snowfall (cm)': 'snowfall',
    'Seasons': 'seasons',
    'Holiday': 'holiday',
    'Functioning Day': 'functioning_day',
}
df = df.rename(columns=COLUMN_NAMES)
df.columns.tolist()

['date',
 'rented_bike_count',
 'hour',
 'temperature',
 'humidity',
 'wind_speed',
 'visibility',
 'dew_point_temp',
 'solar_radiation',
 'rainfall',
 'snowfall',
 'seasons',
 'holiday',
 'functioning_day']

In [3]:
# Date는 dd/mm/yyyy 문자열 -> datetime
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
print(df['date'].min(), '~', df['date'].max())
df.dtypes

2017-12-01 00:00:00 ~ 2018-11-30 00:00:00


date                 datetime64[us]
rented_bike_count             int64
hour                          int64
temperature                 float64
humidity                      int64
wind_speed                  float64
visibility                    int64
dew_point_temp              float64
solar_radiation             float64
rainfall                    float64
snowfall                    float64
seasons                         str
holiday                         str
functioning_day                 str
dtype: object

In [4]:
# 정리본을 UTF-8로 저장 후 다시 읽어서 확인
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False, encoding='utf-8')

check = pd.read_csv(CLEAN_PATH, parse_dates=['date'])
print(check.shape)
print(check.equals(df))

(8760, 14)
True


## 3. EDA

In [ ]:
import matplotlib.pyplot as plt

# 운영하지 않은 시간(functioning_day == 'No')은 대여 수가 전부 0이라 평균을 왜곡하므로
# 수요 관련 그래프는 운영한 시간만으로 그림 (df 자체는 변경하지 않음)
op = df[df['functioning_day'] == 'Yes']
print(df['functioning_day'].value_counts())
print('운영 시간 중 대여 수 0인 행:', (op['rented_bike_count'] == 0).sum())
op.describe()

In [ ]:
# 타깃 분포
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(op['rented_bike_count'], bins=50)
ax.set_xlabel('Rented bike count (per hour)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of hourly rentals')
plt.show()
print('skewness:', round(op['rented_bike_count'].skew(), 3))

In [ ]:
# 시간대별 평균 대여 수
hourly = op.groupby('hour')['rented_bike_count'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hourly.index, hourly.values, marker='o')
ax.set_xticks(range(24))
ax.set_xlabel('Hour')
ax.set_ylabel('Mean rented bike count')
ax.set_title('Mean rentals by hour')
ax.grid(alpha=0.3)
plt.show()
hourly.round(1)

In [ ]:
# 계절별 평균 대여 수
season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
seasonal = op.groupby('seasons')['rented_bike_count'].mean().reindex(season_order)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(seasonal.index, seasonal.values)
ax.set_ylabel('Mean rented bike count')
ax.set_title('Mean rentals by season')
plt.show()
seasonal.round(1)

In [ ]:
# 기온과 대여 수
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(op['temperature'], op['rented_bike_count'], s=3, alpha=0.3)
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Rented bike count')
ax.set_title('Temperature vs rentals')
plt.show()

# 5°C 구간별 평균
temp_bin = pd.cut(op['temperature'], bins=range(-20, 45, 5))
op.groupby(temp_bin, observed=True)['rented_bike_count'].agg(['mean', 'count']).round(1)

In [ ]:
# 비·눈 여부에 따른 평균 대여 수
weather = pd.DataFrame({
    'rain': op.groupby(op['rainfall'] > 0)['rented_bike_count'].mean(),
    'snow': op.groupby(op['snowfall'] > 0)['rented_bike_count'].mean(),
}).rename(index={False: 'No', True: 'Yes'})

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for ax, col in zip(axes, ['rain', 'snow']):
    ax.bar(weather.index, weather[col])
    ax.set_title(f'{col} > 0')
axes[0].set_ylabel('Mean rented bike count')
plt.show()
print('rainfall > 0 시간:', (op['rainfall'] > 0).sum(), '/ snowfall > 0 시간:', (op['snowfall'] > 0).sum())
weather.round(1)

In [ ]:
# 공휴일/평일, 주말/주중의 시간대별 패턴 (요일은 그래프용으로만 계산, df에는 추가하지 않음)
day_type = (op['date'].dt.dayofweek >= 5).map({False: 'Weekday', True: 'Weekend'})

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for label, g in op.groupby('holiday'):
    axes[0].plot(g.groupby('hour')['rented_bike_count'].mean(), marker='o', label=label)
for label, g in op.groupby(day_type):
    axes[1].plot(g.groupby('hour')['rented_bike_count'].mean(), marker='o', label=label)
axes[0].set_title('Holiday vs No Holiday')
axes[1].set_title('Weekday vs Weekend')
for ax in axes:
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel('Hour')
    ax.legend()
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Mean rented bike count')
plt.show()

print(op.groupby('holiday')['rented_bike_count'].mean().round(1))
print(op.groupby(day_type)['rented_bike_count'].mean().round(1))

In [ ]:
# 수치형 변수 간 상관관계
corr = op.select_dtypes('number').corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im)
ax.set_title('Correlation matrix')
plt.tight_layout()
plt.show()
corr['rented_bike_count'].drop('rented_bike_count').sort_values(key=abs, ascending=False).round(3)

In [ ]:
# humidity 최솟값 0 확인 (이상치 여부)
print('humidity == 0 행 수:', (df['humidity'] == 0).sum())
print('humidity 하위 값:', sorted(df['humidity'].unique())[:10])
df[df['humidity'] == 0][['date', 'hour', 'temperature', 'humidity', 'dew_point_temp', 'rainfall', 'rented_bike_count']]

## 4. 전처리

In [ ]:
import numpy as np

# humidity == 0 인 17행은 같은 행의 기온·이슬점과 맞지 않음 (측정 오류로 추정)
# → 기온·이슬점으로 상대습도를 계산(Magnus 공식)해서 대체
def humidity_from_dew_point(temp, dew_point):
    a, b = 17.625, 243.04
    return 100 * np.exp(a * dew_point / (b + dew_point)) / np.exp(a * temp / (b + temp))

zero_humidity = df['humidity'] == 0
estimated = humidity_from_dew_point(df.loc[zero_humidity, 'temperature'], df.loc[zero_humidity, 'dew_point_temp'])
df.loc[zero_humidity, 'humidity'] = estimated.round().astype(int)

print('보정한 행 수:', zero_humidity.sum())
print('보정 후 humidity == 0 행 수:', (df['humidity'] == 0).sum())
df.loc[zero_humidity, ['date', 'hour', 'temperature', 'dew_point_temp', 'humidity']]

In [ ]:
# 운영하지 않은 시간(functioning_day == 'No', 295행)은 대여 수가 전부 0 → 수요가 아니므로 제외
# 제외 후에는 값이 'Yes' 하나뿐이라 컬럼도 삭제
data = df[df['functioning_day'] == 'Yes'].drop(columns='functioning_day').reset_index(drop=True)
print(df.shape, '->', data.shape)

In [ ]:
# 날짜에서 요일(0=월 ~ 6=일), 월, 주말 여부 추출 후 date 삭제
data['day_of_week'] = data['date'].dt.dayofweek
data['month'] = data['date'].dt.month
data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)
data = data.drop(columns='date')
data[['day_of_week', 'month', 'is_weekend']].head()

In [ ]:
# 범주형 인코딩: seasons는 원-핫, holiday는 값이 2개라 0/1 한 컬럼으로 변환
data = pd.get_dummies(data, columns=['seasons'], dtype=int)
data['holiday'] = (data['holiday'] == 'Holiday').astype(int)
print(data.shape)
data.dtypes

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
TARGET = 'rented_bike_count'

X = data.drop(columns=TARGET)
y = data[TARGET]

# 무작위 분할: train 70 / validation 15 / test 15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)

for name, part in [('train', X_train), ('validation', X_val), ('test', X_test)]:
    print(f'{name:10s} {len(part):5d}행 ({len(part) / len(X):.1%})')

In [ ]:
from sklearn.preprocessing import StandardScaler

# 연속형·순서형 변수만 표준화 (0/1 컬럼은 그대로)
# 데이터 누수 방지: train으로만 fit, validation/test는 transform만
binary_cols = ['holiday', 'is_weekend'] + [c for c in X.columns if c.startswith('seasons_')]
scale_cols = [c for c in X.columns if c not in binary_cols]

scaler = StandardScaler()
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols] = scaler.transform(X_val[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print('표준화한 컬럼:', scale_cols)
print('그대로 둔 컬럼:', binary_cols)
X_train[scale_cols].agg(['mean', 'std']).round(3)

## 5. 기본 모델

In [ ]:
import keras

# random, numpy, tensorflow 시드를 한 번에 고정
keras.utils.set_random_seed(SEED)

model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1),  # 회귀: 출력 뉴런 1개, 활성화 함수 없음
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

In [ ]:
# test 데이터는 사용하지 않고 validation으로만 확인
history = model.fit(
    X_train.to_numpy('float32'), y_train.to_numpy('float32'),
    validation_data=(X_val.to_numpy('float32'), y_val.to_numpy('float32')),
    epochs=100,
    batch_size=32,
    verbose=0,
)

In [ ]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key in zip(axes, ['loss', 'mae']):
    ax.plot(history.history[key], label='train')
    ax.plot(history.history[f'val_{key}'], label='validation')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(key.upper())
    ax.set_title(f'Baseline {key.upper()}')
    ax.legend()
    ax.grid(alpha=0.3)
axes[0].set_yscale('log')
plt.show()

In [ ]:
val_mse, val_mae = model.evaluate(X_val.to_numpy('float32'), y_val.to_numpy('float32'), verbose=0)
print(f'validation MSE : {val_mse:,.1f}')
print(f'validation RMSE: {np.sqrt(val_mse):,.1f}')
print(f'validation MAE : {val_mae:,.1f}')

## 6. 하이퍼파라미터 실험

기본 모델에서 한 번에 하나씩만 바꾸고, 각 실험에서 validation MSE가 가장 낮은 값을 다음 실험에 사용 (test 데이터 사용 안 함)

In [ ]:
X_train_np, y_train_np = X_train.to_numpy('float32'), y_train.to_numpy('float32')
X_val_np, y_val_np = X_val.to_numpy('float32'), y_val.to_numpy('float32')

BASE = {'hidden_units': (64, 32), 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 100}
results = []    # 모든 실험 결과
histories = {}  # 학습 곡선용

def build_model(hidden_units, activation, learning_rate):
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential([keras.Input(shape=(X_train_np.shape[1],))])
    for units in hidden_units:
        model.add(keras.layers.Dense(units, activation=activation))
    model.add(keras.layers.Dense(1))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss='mse', metrics=['mae'])
    return model

def run_experiment(experiment, param, values, base):
    """base에서 param 하나만 values로 바꿔가며 학습하고, validation MSE가 가장 낮은 값을 반환"""
    rows = []
    for value in values:
        config = {**base, param: value}
        model = build_model(config['hidden_units'], config['activation'], config['learning_rate'])
        history = model.fit(X_train_np, y_train_np, validation_data=(X_val_np, y_val_np),
                            epochs=config['epochs'], batch_size=config['batch_size'], verbose=0)
        val_mse, val_mae = model.evaluate(X_val_np, y_val_np, verbose=0)
        rows.append({'experiment': experiment, 'param': param, 'value': str(value),
                     'val_MSE': val_mse, 'val_RMSE': np.sqrt(val_mse), 'val_MAE': val_mae,
                     'params': model.count_params()})
        histories[(experiment, str(value))] = history.history
    results.extend(rows)
    table = pd.DataFrame(rows)
    best = values[table['val_MSE'].idxmin()]
    return table, best

def plot_experiment(experiment, values):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for value in values:
        h = histories[(experiment, str(value))]
        axes[0].plot(h['val_loss'], label=str(value))
        axes[1].plot(h['val_mae'], label=str(value))
    axes[0].set_yscale('log')
    for ax, name in zip(axes, ['validation MSE', 'validation MAE']):
        ax.set_xlabel('Epoch')
        ax.set_ylabel(name)
        ax.set_title(f'{experiment}: {name}')
        ax.legend()
        ax.grid(alpha=0.3)
    plt.show()

### 실험 1: layer 수 / 뉴런 수

In [ ]:
units_values = [(32,), (64, 32), (128, 64), (128, 64, 32)]
exp1, best_units = run_experiment('1. layers/units', 'hidden_units', units_values, BASE)
plot_experiment('1. layers/units', units_values)
print('선택:', best_units)
exp1.round(1)

### 실험 2: 활성화 함수

In [ ]:
config2 = {**BASE, 'hidden_units': best_units}
activation_values = ['relu', 'tanh', 'elu']
exp2, best_activation = run_experiment('2. activation', 'activation', activation_values, config2)
plot_experiment('2. activation', activation_values)
print('선택:', best_activation)
exp2.round(1)

### 실험 3: learning rate

In [ ]:
config3 = {**config2, 'activation': best_activation}
lr_values = [0.0001, 0.001, 0.003, 0.01]
exp3, best_lr = run_experiment('3. learning rate', 'learning_rate', lr_values, config3)
plot_experiment('3. learning rate', lr_values)
print('선택:', best_lr)
exp3.round(1)

In [ ]:
# 전체 실험 결과 요약
best_config = {**config3, 'learning_rate': best_lr}
print('현재까지 최적 조합:', best_config)
pd.DataFrame(results).round(1)

### 실험 4: 타깃 로그 변환

타깃을 `log1p`로 변환해 학습하고, 예측값을 `expm1`로 되돌려 원래 단위(대여 대수)에서 비교

In [ ]:
config4 = best_config
target_values = ['raw', 'log1p']
rows = []
for target in target_values:
    model = build_model(config4['hidden_units'], config4['activation'], config4['learning_rate'])
    use_log = target == 'log1p'
    y_fit = np.log1p(y_train_np) if use_log else y_train_np
    y_val_fit = np.log1p(y_val_np) if use_log else y_val_np
    history = model.fit(X_train_np, y_fit, validation_data=(X_val_np, y_val_fit),
                        epochs=config4['epochs'], batch_size=config4['batch_size'], verbose=0)
    histories[('4. target', target)] = history.history

    pred = model.predict(X_val_np, verbose=0).ravel()
    if use_log:
        pred = np.expm1(pred)
    val_mse = float(np.mean((pred - y_val_np) ** 2))
    rows.append({'experiment': '4. target', 'param': 'target', 'value': target,
                 'val_MSE': val_mse, 'val_RMSE': np.sqrt(val_mse), 'val_MAE': float(np.mean(np.abs(pred - y_val_np))),
                 'params': model.count_params(), 'negative_preds': int((pred < 0).sum())})
results.extend(rows)
exp4 = pd.DataFrame(rows)
best_target = target_values[exp4['val_MSE'].idxmin()]

# 학습 곡선 (loss 단위가 다르므로 각각 그림)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, target in zip(axes, target_values):
    h = histories[('4. target', target)]
    ax.plot(h['loss'], label='train')
    ax.plot(h['val_loss'], label='validation')
    ax.set_yscale('log')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(f'MSE ({target} scale)')
    ax.set_title(f'4. target: {target}')
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()
print('선택:', best_target)
exp4.round(1)

In [ ]:
# 전체 실험 결과 요약 (실험 1~4)
best_config = {**config4, 'target': best_target}
print('최적 조합:', best_config)
pd.DataFrame(results).drop(columns='negative_preds').round(1)